# HOG-Based Homography Pipeline

End-to-end pipeline for registering broadcast hockey frames to the bird's-eye rink template: HOG-based reference-frame retrieval, adaptive structural feature extraction, masked SIFT + ECC alignment, and RHO homography estimation. The second half of the notebook reproduces the quantitative evaluation (Table 1 / Figures 3-5) from the project report.

See `../README.md` and `../docs/report.pdf` for full context.

## 1. Imports

In [ ]:
import cv2
import numpy as np
import os
import glob
import time
import matplotlib.pyplot as plt

from skimage.metrics import structural_similarity as ssim


## 2. Configuration

In [ ]:
TEMPLATE_IMAGE_PATH = 'data/template/birdseye_template.png'
TEMPLATE_POINTS_FILE = 'data/template/bird.txt'
TRAIN_IMAGES_DIR = 'data/train/images'
TRAIN_LABELS_DIR = 'data/train/points'
TEST_IMAGES_DIR = 'data/test/images'

## 3. Feature Extraction

Crowd suppression (yellow kick-plate ROI mask) and the adaptive bilateral-filter / CLAHE / adaptive-threshold pipeline that produces the binary rink-line "skeleton".

In [ ]:
def create_roi_mask(img, h, w):
    """
    Generates a binary mask based on the yellow board line.
    """
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Finding Yellow Border Line
    lower_yellow = np.array([20, 100, 100])
    upper_yellow = np.array([35, 255, 255])
    mask_yellow = cv2.inRange(img_hsv, lower_yellow, upper_yellow)

    pts = cv2.findNonZero(mask_yellow)
    if pts is None or len(pts) < 50: return None

    pts = pts.squeeze()
    if pts.ndim == 1: pts = pts.reshape(-1, 2)
    xs, ys = pts[:, 0], pts[:, 1]

    try:
        # Fit a 2nd degree polynomial
        coeff = np.polyfit(xs, ys, 2)
        poly = np.poly1d(coeff)
    except: return None

    roi_mask = np.zeros((h, w), dtype=np.uint8)
    Y, X = np.ogrid[:h, :w]
    curve_y = poly(X)
    roi_mask[Y > curve_y] = 255
    return roi_mask

def extract_features_adaptive(img, visualize=False):
    """
    Extracts the 'Skeleton' of the rink lines.
    """
    if img is None: return None

    # 1. Filter & CLAHE
    img_blur = cv2.bilateralFilter(img, 9, 75, 75)
    lab = cv2.cvtColor(img_blur, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l_eq = clahe.apply(l)

    # 2. Adaptive Threshold
    binary = cv2.adaptiveThreshold(l_eq, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 33, 4)

    # 3. ROI Mask (Remove Crowd)
    h, w = binary.shape
    dynamic_mask = create_roi_mask(img, h, w)
    if dynamic_mask is not None:
        binary = cv2.bitwise_and(binary, binary, mask=dynamic_mask)
    else:
        binary[0:int(h * 0.25), :] = 0

    # 4. Connected Components (Remove Player blobs/Noise)
    num, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    filtered_cc = np.zeros_like(binary)
    for i in range(1, num):
        # Filter small noise (skates, stick tape)
        if stats[i, cv2.CC_STAT_AREA] > 200:
            filtered_cc[labels == i] = 255

    # 5. Dilate (Thicken lines)
    kernel = np.ones((3,3), np.uint8)
    clean_skeleton = cv2.dilate(filtered_cc, kernel, iterations=1)

    if visualize:
        plt.figure(figsize=(10,5))
        plt.imshow(clean_skeleton, cmap='gray')
        plt.title("Extracted Features (Skeleton)")
        plt.axis('off'); plt.show()

    return clean_skeleton


## 4. Reference Database & HOG Retrieval

Builds the HOG-feature database from the training frames and finds the nearest-neighbor reference frame for a given test frame.

In [ ]:
def parse_keypoints(file_path):
    keypoints = {}
    if not os.path.exists(file_path): return keypoints
    with open(file_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        if not line: continue
        parts = line.split()
        try:
            x, y = float(parts[-2]), float(parts[-1])
        except: continue
        raw_name = parts[:-2]
        clean_name = [p for p in raw_name if not (p.startswith('[') or p.endswith(']'))]
        name = "_".join(clean_name)
        if name: keypoints[name] = (x, y)
    return keypoints

def build_database(img_dir, lbl_dir):
    database = []
    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")) + glob.glob(os.path.join(img_dir, "*.png")))
    print(f"Building database from {len(image_paths)} images...")
    
    hog = cv2.HOGDescriptor()
    
    for i, img_path in enumerate(image_paths):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(lbl_dir, base_name + ".txt")
        if not os.path.exists(lbl_path): continue
            
        img = cv2.imread(img_path)
        if img is None: continue
        
        # Pre-calculate Skeleton here
        skeleton = extract_features_adaptive(img, visualize=False)

        small = cv2.resize(img, (64, 128))
        h = hog.compute(small).flatten()
        
        database.append({
            "name": base_name,
            "hog": h,
            "keypoints": parse_keypoints(lbl_path),
            "image": img,
            "skeleton": skeleton 
        })
    return database

def find_best_match(test_img, database):
    hog = cv2.HOGDescriptor()
    small = cv2.resize(test_img, (64, 128))
    test_hog = hog.compute(small).flatten()
    best_match, min_dist = None, float('inf')
    for record in database:
        dist = np.linalg.norm(test_hog - record['hog'])
        if dist < min_dist:
            min_dist = dist
            best_match = record
    return best_match


## 5. Alignment: Masked SIFT + ECC Refinement

In [ ]:
def refine_alignment_ecc(template_skeleton, test_skeleton, initial_warp):
    """
    Uses ECC to snap the training skeleton onto the test skeleton.
    """
    img1 = cv2.GaussianBlur(template_skeleton, (5, 5), 0).astype(np.float32) / 255.0
    img2 = cv2.GaussianBlur(test_skeleton, (5, 5), 0).astype(np.float32) / 255.0
    
    warp_mode = cv2.MOTION_HOMOGRAPHY
    warp_matrix = initial_warp.astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 500, 1e-5)

    print("Running ECC Refinement...")
    try:
        (cc, warp_matrix) = cv2.findTransformECC(img1, img2, warp_matrix, warp_mode, criteria)
        print(f"ECC Converged. Correlation: {cc:.4f}")
        return warp_matrix
    except cv2.error:
        print(f"ECC Failed to converge.")
        return initial_warp


def transfer_keypoints_smart(test_img, best_match, template_points):
    train_img = best_match['image']
    train_skel = best_match['skeleton']
    train_kps_dict = best_match['keypoints']

    # 1. Extract Skeleton (Raw Image)
    print("Extracting features...")
    test_skel = extract_features_adaptive(test_img, visualize=False)

    # 2. Masked SIFT (Using your RGB Mask Idea)
    masked_train_rgb = cv2.bitwise_and(train_img, train_img, mask=train_skel)
    masked_test_rgb = cv2.bitwise_and(test_img, test_img, mask=test_skel)

    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(masked_train_rgb, None)
    kp2, des2 = sift.detectAndCompute(masked_test_rgb, None)

    if kp1 is None or kp2 is None or len(kp1) < 5 or len(kp2) < 5: return [], [], None

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)
    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance: good.append(m)

    if len(good) < 4: return [], [], None

    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    # 3. INITIAL ALIGNMENT (Affine Physics Constraint)
    # affine_matrix, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts)
    affine_matrix = None

    if affine_matrix is None:
        H_init, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    else:
        H_init = np.vstack([affine_matrix, [0, 0, 1]])

    # 4. REFINEMENT (ECC)
    # Refine the match using the skeletons
    H_refined = refine_alignment_ecc(train_skel, test_skel, H_init)

    # 5. Transfer Points
    common = [k for k in train_kps_dict if k in template_points]
    if not common: return [], [], None

    pts_to_transfer = np.float32([train_kps_dict[k] for k in common]).reshape(-1, 1, 2)
    transferred = cv2.perspectiveTransform(pts_to_transfer, H_refined)

    final_src = transferred.reshape(-1, 2)
    return final_src, np.float32([template_points[k] for k in common]), H_refined


## 6. Visualization Helpers

In [ ]:
def visualize_match(test_img, match_img, match_name, test_name):
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)); plt.title(f"Test: {test_name}"); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB)); plt.title(f"Match: {match_name}"); plt.axis('off')
    plt.tight_layout(); plt.show()

def visualize_test_on_train_overlay(test_img, train_img, H_train_to_test):
    """
    Warps the Test Image BACK onto the Training Image.
    This shows if the FIRST STEP (Transfer) worked correctly.
    """
    try:
        H_inv = np.linalg.inv(H_train_to_test)
    except: return
    h, w = train_img.shape[:2]
    warped_test = cv2.warpPerspective(test_img, H_inv, (w, h))
    
    gray_warped = cv2.cvtColor(warped_test, cv2.COLOR_BGR2GRAY)
    mask = cv2.threshold(gray_warped, 1, 255, cv2.THRESH_BINARY)[1]
    
    train_rgb = cv2.cvtColor(train_img, cv2.COLOR_BGR2RGB)
    warped_rgb = cv2.cvtColor(warped_test, cv2.COLOR_BGR2RGB)
    
    overlay = train_rgb.copy()
    overlay[mask==255] = cv2.addWeighted(train_rgb[mask==255], 0.5, warped_rgb[mask==255], 0.5, 0)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(overlay); plt.title("ALIGNMENT CHECK: Test Warped onto Train"); plt.axis('off'); plt.show()

def visualize_birdseye_overlay(test_img, template_path, H, img_name):
    if not os.path.exists(template_path): return
    template = cv2.cvtColor(cv2.imread(template_path), cv2.COLOR_BGR2RGB)
    h, w = template.shape[:2]
    warped = cv2.warpPerspective(test_img, H, (w, h))
    warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
    
    mask = cv2.threshold(cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY), 1, 255, cv2.THRESH_BINARY)[1]
    
    template_faded = (template * 0.3).astype(np.uint8)
    overlay = template_faded.copy()
    overlay[mask==255] = cv2.addWeighted(template[mask==255], 0.2, warped_rgb[mask==255], 0.8, 0)
    base_name = os.path.basename(img_name)
    print(base_name)
    save_path = os.path.join("wraps", f"{base_name}")
    cv2.imwrite(save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    
    plt.imshow(overlay); plt.title(img_name); plt.axis('off'); plt.show()


## 7. Run Pipeline on Test Set

In [ ]:
print("Loading Template Points...")
template_dict = parse_keypoints(TEMPLATE_POINTS_FILE)

db = build_database(TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR)

test_H = {}
times = {}
for f in os.listdir(TEST_IMAGES_DIR):
    if f.endswith('.png') or f.endswith('.jpg'):
        img_path = os.path.join(TEST_IMAGES_DIR, f)
        print(f"\n--- Processing {f} ---")
        
        test_img = cv2.imread(img_path)
        if not db or test_img is None: continue

        # A. Find Match
        best_match = find_best_match(test_img, db)
        print(f"Closest Training Frame: {best_match['name']}")
        # visualize_match(test_img, best_match['image'], best_match['name'], f)

        # Start counting time
        start = time.perf_counter()
        
        # B. Smart Transfer
        src, dst, H_transfer = transfer_keypoints_smart(test_img, best_match, template_dict)

        if H_transfer is not None and len(src) >= 4:
            # C. Check Alignment (Test overlayed on Train)
            # visualize_test_on_train_overlay(test_img, best_match['image'], H_transfer)

            # D. Final Homography (Using RHO)
            H_final, _ = cv2.findHomography(src, dst, cv2.RHO, 3.0)
            
            # E. Final Result
            # visualize_birdseye_overlay(test_img, TEMPLATE_IMAGE_PATH, H_final, img_path)

            # F. store image and final H
            test_H[img_path] = H_final
        else:
            print("Failed to transfer points.")

        # End counting time 
        end = time.perf_counter()
        times[img_path] = end - start 


## 8. Evaluation

### 8.1 Ground-truth template -> bird's-eye homography

In [ ]:
template_img = cv2.imread("data/template/template4.png")
template_img = cv2.cvtColor(template_img, cv2.COLOR_BGR2RGB)

bird_img = cv2.imread("data/template/birdseye_template.png")
bird_img = cv2.cvtColor(bird_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,3))
plt.subplot(1,2,1)
plt.imshow(template_img)
plt.title("template4")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(bird_img)
plt.title("bird-eye template")
plt.axis("off")

In [ ]:
def load_points_txt(path):
    pts = {}
    with open(path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            name, x, y = line.split()
            pts[name] = (float(x), float(y))
    return pts


template_pts = load_points_txt("data/template/template4_points.txt")
bird_pts     = load_points_txt("data/template/birdseye_template_points.txt")

print("template points:", len(template_pts))
print("bird-eye points:", len(bird_pts))

In [ ]:
common_keys = sorted(set(template_pts.keys()) & set(bird_pts.keys()))

In [ ]:
src_pts = np.array(
    [template_pts[k] for k in common_keys],
    dtype=np.float32
)

dst_pts = np.array(
    [bird_pts[k] for k in common_keys],
    dtype=np.float32
)

In [ ]:
H, mask = cv2.findHomography(
    src_pts,
    dst_pts,
    method=cv2.RANSAC,
    ransacReprojThreshold=5.0
)

In [ ]:
h, w, _ = bird_img.shape

warped = cv2.warpPerspective(
    template_img,
    H,
    (w, h)
)

overlay = cv2.addWeighted(bird_img, 0.6, warped, 0.4, 0)

plt.figure(figsize=(6,4))
plt.imshow(overlay)
plt.title("Warped template → bird-eye (overlay)")
plt.axis("off")

### 8.2 Ground-truth test -> bird's-eye homographies

In [ ]:
test_img_dir   = "data/test/images"
test_pts_dir   = "data/test/points"


In [ ]:
image_files = sorted([
    f for f in os.listdir(test_img_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print("Test images:", len(image_files))

In [ ]:
H_test_GD = {}

for img_name in image_files:

    print(f"\nProcessing {img_name}")

    img_path = os.path.join(test_img_dir, img_name)
    pts_path = os.path.join(
        test_pts_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(pts_path):
        print("  points file missing, skip")
        continue

    # --- load image ---
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- load test points ---
    test_pts = load_points_txt(pts_path)

    # --- find common points ---
    common_keys = sorted(
        set(test_pts.keys()) & set(template_pts.keys())
    )

    if len(common_keys) < 4:
        print("  not enough correspondences, skip")
        continue

    src_pts = np.array(
        [test_pts[k] for k in common_keys],
        dtype=np.float32
    )

    mid_pts = np.array(
        [template_pts[k] for k in common_keys],
        dtype=np.float32
    )

    # --- H: test -> template4 ---
    H_test_to_template, mask = cv2.findHomography(
        src_pts,
        mid_pts,
        cv2.RANSAC,
        5.0
    )

    if H_test_to_template is None:
        print("  homography failed")
        continue

    # --- compose H: test -> bird-eye ---
    H_test_to_be = H @ H_test_to_template

    # --- warp ---
    h_be, w_be, _ = bird_img.shape
    warped = cv2.warpPerspective(
        img,
        H_test_to_be,
        (w_be, h_be)
    )

    overlay = cv2.addWeighted(
        bird_img, 0.6,
        warped,   0.4,
        0
    )

    H_test_GD[img_path] = H_test_to_be

    # --- display ---
    # plt.figure(figsize=(8,4))
    # plt.imshow(overlay)
    # plt.title(f"{img_name} → bird-eye overlay")
    # plt.axis("off")

    # plt.show()

### 8.3 Pixel-Level Comparison (L1 / L2 / SSIM)

In [ ]:
def warp_to_birdeye(img, H, bird_shape):
    h, w = bird_shape[:2]
    return cv2.warpPerspective(img, H, (w, h))

In [ ]:
def pixel_error(img1, img2, mask=None):
    diff = img1.astype(np.float32) - img2.astype(np.float32)
    if mask is not None:
        diff = diff[mask]
    l1 = np.mean(np.abs(diff))
    l2 = np.sqrt(np.mean(diff**2))
    return l1, l2

In [ ]:
def ssim_score(img1, img2, mask=None):
    g1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)

    if mask is not None:
        return ssim(g1, g2, data_range=255)
    else:
        return ssim(g1, g2, data_range=255)

In [ ]:
def valid_warp_mask(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return gray > 0

In [ ]:
results = []

for img_path in H_test_GD.keys():

    if img_path not in test_H:
        continue

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # warp
    warp_gt   = warp_to_birdeye(img, H_test_GD[img_path], bird_img.shape)
    warp_pred = warp_to_birdeye(img, test_H[img_path], bird_img.shape)

    # valid region
    mask = valid_warp_mask(warp_gt) & valid_warp_mask(warp_pred)

    # metrics
    l1, l2 = pixel_error(warp_gt, warp_pred, mask)
    s      = ssim_score(warp_gt, warp_pred)

    results.append({
        "image": img_path,
        "L1": l1,
        "L2": l2,
        "SSIM": s
    })

print("Evaluated images:", len(results))

In [ ]:
L1_vals   = [r["L1"] for r in results]
L2_vals   = [r["L2"] for r in results]
SSIM_vals = [r["SSIM"] for r in results]

print("Mean L1 :", np.mean(L1_vals))
print("Mean L2 :", np.mean(L2_vals))
print("Mean SSIM:", np.mean(SSIM_vals))

In [ ]:
def visualize_difference(warp_gt, warp_pred):
    diff = cv2.absdiff(warp_gt, warp_pred)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
    diff_norm = cv2.normalize(diff_gray, None, 0, 255, cv2.NORM_MINMAX)
    return diff_norm

In [ ]:
example = results[0]["image"]

img = cv2.imread(example)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

warp_gt   = warp_to_birdeye(img, H_test_GD[example], bird_img.shape)
warp_pred = warp_to_birdeye(img, test_H[example], bird_img.shape)

diff_vis = visualize_difference(warp_gt, warp_pred)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(warp_gt)
plt.title("GT warp")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(warp_pred)
plt.title("Pred warp")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(diff_vis, cmap="hot")
plt.title("Difference")
plt.axis("off")

### 8.4 Reprojection Error (Table 1)

In [ ]:
def project_points(pts, H):
    """
    pts: (N,2)
    H:   (3,3)
    """
    pts = np.asarray(pts, dtype=np.float64)

    pts_h = np.hstack([
        pts,
        np.ones((pts.shape[0], 1))
    ])  # (N,3)

    proj = (H @ pts_h.T).T
    proj = proj[:, :2] / proj[:, 2:3]

    return proj

In [ ]:
def reprojection_error_between_H(pts, H_gt, H_pred):
    """
    pts: (N,2) test image points
    """
    proj_gt   = project_points(pts, H_gt)
    proj_pred = project_points(pts, H_pred)

    errors = np.linalg.norm(proj_gt - proj_pred, axis=1)
    return errors

In [ ]:
all_errors = []
each_errors = {}

for img_path in H_test_GD.keys():

    if img_path not in test_H:
        continue

    pts_path = os.path.join(
        "data/test/points",
        os.path.splitext(os.path.basename(img_path))[0] + ".txt"
    )

    if not os.path.exists(pts_path):
        continue

    test_pts_dict = load_points_txt(pts_path)
    test_pts = np.array(list(test_pts_dict.values()), dtype=np.float64)

    errs = reprojection_error_between_H(
        test_pts,
        H_test_GD[img_path],
        test_H[img_path]
    )



    each_errors[img_path] = errs
    all_errors.extend(errs)

In [ ]:
all_errors = np.array(all_errors)

print("Reprojection error (px)")
print("Mean  :", all_errors.mean())
print("Median:", np.median(all_errors))
print("Std   :", all_errors.std())
print("Max   :", all_errors.max())

In [ ]:
for img_path in each_errors.keys():
    print(img_path)
    print("Reprojection error (px)")
    print("Mean  :", each_errors[img_path].mean())
    print("Median:", np.median(each_errors[img_path]))
    print("Std   :", each_errors[img_path].std())
    print("Max   :", each_errors[img_path].max())

### 8.5 Runtime per Image (Figure 5)

In [ ]:
for img in times.keys():
    print(img + ": " + str(times[img]))